In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# Wild Animal Species Classification

**Task**: Classify images into 90 wild animal species.

**Dataset**: [Animal Image Dataset (90 Different Animals)](https://www.kaggle.com/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals)
- 5,400 images across 90 classes (60 images per class)
- ~688 MB
- All images sourced from Google Images

**Approach**:
- **Architecture 1 — Custom CNN**: designed from scratch, 90-class output
- **Architecture 2 — EfficientNetB0**: pre-trained on ImageNet, fine-tuned on animal species

**Download the dataset from Kaggle and place it so the path below is correct.**

### Imports

In [ ]:
import gc
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
%matplotlib inline

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from tensorflow.keras import backend as K
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (Input, Conv2D, Dense, Dropout,
                                      Flatten, MaxPool2D, BatchNormalization,
                                      GlobalAveragePooling2D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.random import set_seed

print('TensorFlow version:', tf.__version__)

### Paths — update this to where you placed the dataset

In [ ]:
# The dataset has one folder per animal: animals/animals/<species>/
DATA_DIR = '/content/drive/MyDrive/datasets/wild_animals/animals/animals'

IMG_SIZE   = 224   # EfficientNet standard size, good for custom CNN too
BATCH_SIZE = 32
EPOCHS     = 50
classes    = 90

print('Data dir exists:', os.path.exists(DATA_DIR))
if os.path.exists(DATA_DIR):
    species = sorted(os.listdir(DATA_DIR))
    print(f'Species found: {len(species)}')
    print('First 10:', species[:10])

### Data loading

The dataset has **no pre-defined train/test split**, so we split manually:
- 80% training (with augmentation)
- 10% validation
- 10% test

We use `ImageDataGenerator` with an 80/20 split, then evaluate on a held-out test set.

In [ ]:
np.random.seed(1402)
set_seed(1981)

# For custom CNN: rescale to [0, 1]
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=1402
)

val_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=1402,
    shuffle=False
)

class_names = list(train_gen.class_indices.keys())
print(f'Train samples : {train_gen.samples}')
print(f'Val   samples : {val_gen.samples}')
print(f'Classes       : {len(class_names)}')

### Visualise samples

In [ ]:
imgs, labels = next(train_gen)
label_map = {v: k for k, v in train_gen.class_indices.items()}

plt.style.use('dark_background')
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(label_map[np.argmax(labels[i])], fontsize=8)
    ax.axis('off')
plt.suptitle('Wild Animals — sample images', fontsize=14)
plt.tight_layout()
plt.show()

### Helper functions

In [ ]:
def plot_history(hs, metric):
    plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = [15, 6]
    plt.rcParams['font.size'] = 14
    plt.clf()
    for label, hist in hs.items():
        plt.plot(hist.history[metric],
                 label=f'{label} train {metric}', linewidth=2)
        plt.plot(hist.history[f'val_{metric}'],
                 label=f'{label} val {metric}', linewidth=2)
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric == 'loss' else 'Accuracy')
    plt.legend()
    plt.show()


def plot_confusion_matrix(model, generator, title='Confusion Matrix'):
    generator.reset()
    y_pred = np.argmax(model.predict(generator, verbose=1), axis=1)
    y_true = generator.classes
    cm = confusion_matrix(y_true, y_pred)
    # For 90 classes the full matrix is huge — plot a summary heatmap
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(20, 16))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    plt.tight_layout()
    plt.show()
    print(classification_report(y_true, y_pred,
                                 target_names=class_names, zero_division=0))


def clean_up(model):
    K.clear_session()
    del model
    gc.collect()

---
## Architecture 1 — Custom CNN

```
Input(224,224,3)
  → Conv2D(32,  3×3) → BN → MaxPool(2×2) → Dropout(0.2)
  → Conv2D(64,  3×3) → BN → MaxPool(2×2) → Dropout(0.2)
  → Conv2D(128, 3×3) → BN → MaxPool(2×2) → Dropout(0.2)
  → Conv2D(256, 3×3) → BN → MaxPool(2×2) → Dropout(0.2)
  → GlobalAveragePooling2D
  → Dense(512, ReLU) → Dropout(0.4)
  → Output(90, softmax)
```

GlobalAveragePooling2D instead of Flatten — reduces parameters and overfitting on a small dataset.

In [ ]:
def build_custom_cnn(dropout=True, batchnorm=True):
    np.random.seed(1402)
    set_seed(1981)

    inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='Input')
    x = inp
    for i, filters in enumerate([32, 64, 128, 256]):
        x = Conv2D(filters, (3,3), padding='same',
                   activation='relu', name=f'Conv-{i+1}')(x)
        if batchnorm:
            x = BatchNormalization(name=f'BN-{i+1}')(x)
        x = MaxPool2D((2,2), name=f'Pool-{i+1}')(x)
        if dropout:
            x = Dropout(0.2, name=f'Drop-{i+1}')(x)

    x = GlobalAveragePooling2D(name='GAP')(x)
    x = Dense(512, activation='relu', name='FC')(x)
    if dropout:
        x = Dropout(0.4, name='Drop-FC')(x)
    out = Dense(classes, activation='softmax', name='Output')(x)
    return Model(inputs=inp, outputs=out)

### Custom CNN — Experiment 1a: No regularisation (baseline)

In [ ]:
cnn_base = build_custom_cnn(dropout=False, batchnorm=False)
cnn_base.compile(optimizer=Adam(),
                 loss='categorical_crossentropy', metrics=['accuracy'])
cnn_base.summary()

cnn_base_hs = cnn_base.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, verbose=1
)

cnn_base_eval = cnn_base.evaluate(val_gen, verbose=1)
print(f'\nVal Acc: {cnn_base_eval[1]:.5f}  Val Loss: {cnn_base_eval[0]:.5f}')
plot_history({'Custom CNN (base)': cnn_base_hs}, 'loss')
plot_history({'Custom CNN (base)': cnn_base_hs}, 'accuracy')
clean_up(cnn_base)

### Custom CNN — Experiment 1b: BatchNorm + Dropout + EarlyStopping + ReduceLR

In [ ]:
es  = EarlyStopping(monitor='val_accuracy', patience=10,
                    verbose=1, restore_best_weights=True)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                        patience=5, verbose=1, min_lr=1e-6)

cnn_reg = build_custom_cnn(dropout=True, batchnorm=True)
cnn_reg.compile(optimizer=Adam(),
                loss='categorical_crossentropy', metrics=['accuracy'])
cnn_reg.summary()

cnn_reg_hs = cnn_reg.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=[es, rlr], verbose=1
)

cnn_reg_eval = cnn_reg.evaluate(val_gen, verbose=1)
ep = len(cnn_reg_hs.history['loss'])
print(f'\nStopped @ epoch {ep}')
print(f'Val Acc: {cnn_reg_eval[1]:.5f}  Val Loss: {cnn_reg_eval[0]:.5f}')
plot_history({'Custom CNN (base)': cnn_base_hs,
              'Custom CNN (reg)':  cnn_reg_hs}, 'loss')
plot_history({'Custom CNN (base)': cnn_base_hs,
              'Custom CNN (reg)':  cnn_reg_hs}, 'accuracy')

### Custom CNN — Confusion Matrix

In [ ]:
plot_confusion_matrix(cnn_reg, val_gen,
                      title='Confusion Matrix — Custom CNN (90 species)')
clean_up(cnn_reg)

---
## Architecture 2 — EfficientNetB0 Transfer Learning

EfficientNetB0 pre-trained on ImageNet. Two-phase training:
1. **Phase 1** (frozen base, 20 epochs): head only
2. **Phase 2** (full fine-tune, up to 30 epochs): all layers, lr=1e-5

In [ ]:
# EfficientNet generators (preprocess_input instead of rescale)
eff_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.15
)

eff_train_gen = eff_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', seed=1402
)
eff_val_gen = eff_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', seed=1402, shuffle=False
)

In [ ]:
np.random.seed(1402)
set_seed(1981)

base = EfficientNetB0(include_top=False, weights='imagenet',
                      input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling='avg')

x = base.output
x = Dense(512, activation='relu',  name='Hidden-1')(x)
x = Dropout(0.3,                   name='Drop-1')(x)
x = Dense(256, activation='relu',  name='Hidden-2')(x)
x = Dropout(0.3,                   name='Drop-2')(x)
out = Dense(classes, activation='softmax', name='Output')(x)
eff_model = Model(inputs=base.input, outputs=out)

# Phase 1 — frozen base
for layer in base.layers:
    layer.trainable = False

eff_model.compile(optimizer=Adam(),
                  loss='categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_accuracy', patience=5,
                   verbose=1, restore_best_weights=True)

eff_upper_hs = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen,
    epochs=20, callbacks=[es], verbose=1
)
print(f'Phase 1 done ({len(eff_upper_hs.history["loss"])} epochs)')

In [ ]:
# Phase 2 — full fine-tune
for layer in base.layers:
    layer.trainable = True

eff_model.compile(optimizer=Adam(learning_rate=1e-5),
                  loss='categorical_crossentropy', metrics=['accuracy'])

es  = EarlyStopping(monitor='val_accuracy', patience=10,
                    verbose=1, restore_best_weights=True)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                        patience=5, verbose=1, min_lr=1e-7)

eff_full_hs = eff_model.fit(
    eff_train_gen, validation_data=eff_val_gen,
    epochs=30, callbacks=[es, rlr], verbose=1
)
print(f'Phase 2 done ({len(eff_full_hs.history["loss"])} epochs)')

eff_eval = eff_model.evaluate(eff_val_gen, verbose=1)
print(f'\nVal Acc: {eff_eval[1]:.5f}  Val Loss: {eff_eval[0]:.5f}')

In [ ]:
upper_ep = len(eff_upper_hs.history['loss'])
full_ep  = len(eff_full_hs.history['loss'])

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = [15, 6]
plt.rcParams['font.size'] = 14
for metric in ['accuracy', 'loss']:
    plt.clf()
    xs1 = np.arange(1, upper_ep + 1)
    xs2 = np.arange(upper_ep + 1, upper_ep + full_ep + 1)
    plt.plot(xs1, eff_upper_hs.history[metric],          label='Phase 1 train', linewidth=2)
    plt.plot(xs1, eff_upper_hs.history[f'val_{metric}'], label='Phase 1 val',   linewidth=2, linestyle='--')
    plt.plot(xs2, eff_full_hs.history[metric],           label='Phase 2 train', linewidth=2)
    plt.plot(xs2, eff_full_hs.history[f'val_{metric}'],  label='Phase 2 val',   linewidth=2, linestyle='--')
    plt.axvline(x=upper_ep, color='white', linestyle=':', alpha=0.5, label='Fine-tune start')
    plt.xlabel('Epochs')
    plt.ylabel('Loss' if metric == 'loss' else 'Accuracy')
    plt.title(f'EfficientNetB0 — {metric}')
    plt.legend()
    plt.show()

### EfficientNetB0 — Confusion Matrix

In [ ]:
plot_confusion_matrix(eff_model, eff_val_gen,
                      title='Confusion Matrix — EfficientNetB0 (90 species)')
clean_up(eff_model)

---
## Final comparison

In [ ]:
print(f"{'Model':<45} {'Val Acc':>10} {'Val Loss':>12}")
print('-' * 69)
rows = [
    ('Custom CNN (no regularisation)',   cnn_base_eval),
    ('Custom CNN (BN + Dropout + ES)',   cnn_reg_eval),
    ('EfficientNetB0 Transfer Learning', eff_eval),
]
for name, ev in rows:
    print(f"{name:<45} {ev[1]:>10.5f} {ev[0]:>12.5f}")